# Import Important Libraries:

In [ ]:
import  pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, MaxPooling2D, Conv2D, Dropout, Flatten

# Import the data or Download it:

In [ ]:
# EMNIST is provided by TensorFlow Datasets 
!pip install -q tensorflow-datasets

In [ ]:
import tensorflow_datasets as tfds

In [ ]:
ds_train = tfds.load(
    "emnist/letters",
    split="train",
    as_supervised=True
)

ds_test = tfds.load(
    "emnist/letters",
    split="test",
    as_supervised=True
)

# Convert data into usable numpy arrays:


In [ ]:
import numpy as np

X_train = []
y_train = []

for image, label in tfds.as_numpy(ds_train):
    X_train.append(image)
    y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(X_train.shape)
print(y_train.shape)

In [ ]:

# Test Data
X_test = []
y_test = []

for image, label in tfds.as_numpy(ds_test):
    X_test.append(image)
    y_test.append(label)

X_test = np.array(X_test)
y_test = np.array(y_test)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

In [ ]:
# TFDS preserves the original EMNIST Letters ids (1..26).
# Guard the conversion so this cell is safe to rerun in Colab.
if y_train.min() == 1 and y_train.max() == 26:
    y_train = y_train - 1
if y_test.min() == 1 and y_test.max() == 26:
    y_test = y_test - 1

assert y_train.min() == 0 and y_train.max() == 25
assert y_test.min() == 0 and y_test.max() == 25
print("Labels:", y_train.min(), "to", y_train.max())

# Let's observe our data:

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(X_train[0:4].squeeze(), cmap="gray")
plt.title(y_train[0:4])
plt.axis("off")
plt.show()

# Fix the EMNIST image orientation:

In [ ]:
X_train = np.transpose(
    X_train,
    axes=(0, 2, 1, 3)
)

X_test = np.transpose(
    X_test,
    axes=(0, 2, 1, 3)
)

# Preprocess the Images:

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0

X_train = X_train.reshape(-1,28,28,1)

X_test = X_test.reshape(-1,28,28,1)


# Building the model:

In [ ]:
def my_model():
  model = Sequential([
      Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
      MaxPooling2D((2,2)),
      Conv2D(64,(3,3),activation='relu'),
      MaxPooling2D((2,2)),
      Flatten(),
      Dense(128,activation='relu'),
      Dropout(0.3),
      Dense(26,activation='softmax'),
  ])

  return model
model  = my_model()

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
# We will use early stopping to save computation and achieve best reults:
from tensorflow.keras.callbacks import EarlyStopping
cbks = EarlyStopping(
    monitor ='val_loss',
    patience = 3,
    restore_best_weights = True,
    verbose=1,
)

In [ ]:
# Re-initialize and compile the model
model = my_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, validation_split=0.2, epochs=20, callbacks=cbks, batch_size=64)

# Save the exact model that was just trained.
model.save("emnist_cnn.keras")

In [ ]:
print('Freshly trained model saved as emnist_cnn.keras')

# Evaluate on test data:

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Make Predictions:

In [ ]:
y_pred_probs = model.predict(X_test)

y_pred = y_pred_probs.argmax(axis=1)

In [ ]:
letters = [
    chr(i)
    for i in range(65,91)
]

# Visualize Predictions:

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

for i in range(10):

    plt.subplot(2,5,i+1)

    plt.imshow(
        X_test[i].squeeze(),
        cmap='gray'
    )

    plt.title(
        f"P:{letters[y_pred[i]]}\nA:{letters[y_test[i]]}"
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

# Building a Confusion Matrix:

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

In [ ]:
import seaborn as sns

plt.figure(figsize=(18,15))

sns.heatmap(
    cm,
    cmap='Blues'
)

plt.title(
    "EMNIST Confusion Matrix"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# Benchmarks Testing:

In [ ]:
from sklearn.metrics import classification_report

# Get the unique classes actually present in the predictions/test set
unique_labels = np.unique(np.concatenate([y_test, y_pred]))
current_target_names = [letters[i] for i in unique_labels]

print(
    classification_report(
        y_test,
        y_pred,
        labels=unique_labels,
        target_names=current_target_names
    )
)

# Check for Errors:

In [ ]:
wrong = np.where(
    y_pred != y_test
)[0]

print(
    "Wrong Predictions:",
    len(wrong)
)

# Display  Mistakes:

In [ ]:
plt.figure(figsize=(15,10))

for i in range(12):

    idx = wrong[i]

    plt.subplot(3,4,i+1)

    plt.imshow(
        X_test[idx].squeeze(),
        cmap='gray'
    )

    plt.title(
        f"P:{letters[y_pred[idx]]}\nA:{letters[y_test[idx]]}"
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

# Save and Download the model:

In [ ]:
model.save("emnist_cnn.keras")

from google.colab import files

files.download(
    "emnist_cnn.keras"
)